# OpenSR-SRGAN RGB-NIR inference

Run the pretrained ESA OpenSR RGB-NIR model on a Sentinel-2 GeoTIFF in Google Colab. The expected GEE export layout is `B2, B3, B4, B8, B11, NDVI, NDRE, SI`; the model receives `B4, B3, B2, B8` in that order.

This notebook writes a georeferenced 4x super-resolved GeoTIFF beside the input. A GPU runtime is recommended, but CPU inference is supported.

## 1. Mount Google Drive

Enable a GPU in **Runtime > Change runtime type** before running the inference cell. Update `INPUT_TIF` in the configuration cell below to point to your GeoTIFF.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

This uses the same dependencies as `requirements-srgan.txt`.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'numpy>=1.26',
    'rasterio>=1.3.9',
    'torch>=2.0',
    'opensr_srgan',
    'opensr-utils',
])
print('SRGAN dependencies installed.')

## 3. Configure the run

The GEE export uses 1-based raster band numbers. For the documented layout, bands 3, 2, 1, and 4 correspond to Red, Green, Blue, and NIR. Set `SCALE_FACTOR = 1.0` when the input is already scaled to 0-1 reflectance.

In [ ]:
from pathlib import Path

INPUT_TIF = '/content/drive/MyDrive/farm_patches/field_01.tif'
OUTPUT_TIF = None  # None creates an _SR.tif file beside INPUT_TIF
SCALE_FACTOR = 10000.0
MODEL_BAND_INDICES = (3, 2, 1, 4)  # 1-based: B4, B3, B2, B8
MODEL_BAND_NAMES = ('Red (B4)', 'Green (B3)', 'Blue (B2)', 'NIR (B8)')

input_path = Path(INPUT_TIF)
if OUTPUT_TIF is None:
    OUTPUT_TIF = str(input_path.with_name(f'{input_path.stem}_SR{input_path.suffix}'))

print(f'Input : {INPUT_TIF}')
print(f'Output: {OUTPUT_TIF}')

In [ ]:
import numpy as np
import rasterio
import torch

def inspect_geotiff(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Input GeoTIFF not found: {path}')

    with rasterio.open(path) as src:
        data = src.read(masked=True)
        values = data.compressed()
        print(f'File: {path}')
        print(f'  Band count : {src.count}')
        print(f'  Dtype      : {src.dtypes[0]}')
        print(f'  CRS        : {src.crs}')
        print(f'  Size       : {src.width} x {src.height}')
        if values.size:
            print(f'  Value range: min={values.min()}, max={values.max()}, mean={values.mean():.2f}')
        else:
            print('  Value range: no unmasked pixels')

        if src.count < max(MODEL_BAND_INDICES):
            raise ValueError(
                f'Expected at least {max(MODEL_BAND_INDICES)} bands for the configured selection; found {src.count}'
            )
        if src.crs is None:
            raise ValueError('Input GeoTIFF has no CRS')
        if not values.size:
            raise ValueError('Input GeoTIFF contains no unmasked pixels')
        if not np.isfinite(values).all():
            raise ValueError('Input GeoTIFF contains non-finite values')

        model_band_labels = ', '.join(MODEL_BAND_NAMES)
        print(f'  Model bands: {MODEL_BAND_INDICES} ({model_band_labels})')
        if values.max() > 20:
            print(f'  Values look like raw digital numbers; normalization will divide by {SCALE_FACTOR}.')
        else:
            print('  Values look like 0-1 reflectance; use SCALE_FACTOR = 1.0 if appropriate.')

inspect_geotiff(INPUT_TIF)

## 4. Prepare model input and load the pretrained model

In [ ]:
from opensr_srgan import load_inference_model

def read_model_input(path):
    with rasterio.open(path) as src:
        arr = src.read(MODEL_BAND_INDICES).astype(np.float32)

    if np.nanmax(arr) > 20:
        if SCALE_FACTOR <= 0:
            raise ValueError('SCALE_FACTOR must be positive')
        arr /= SCALE_FACTOR
    return np.clip(np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0), 0.0, 1.0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    print('WARNING: no GPU detected. Inference will be slower.')
else:
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')

print('Loading pretrained RGB-NIR SRGAN model...')
model = load_inference_model('RGB-NIR').to(device).eval()
model_input = read_model_input(INPUT_TIF)
print(f'Model input shape: {model_input.shape}')

## 5. Run inference and write the georeferenced output

In [ ]:
def run_inference(input_tif, output_tif, model_input, model, device):
    with rasterio.open(input_tif) as src:
        output_profile = src.profile.copy()
        source_transform = src.transform
        source_width, source_height = src.width, src.height

    tensor = torch.from_numpy(model_input).unsqueeze(0).to(device)
    with torch.inference_mode():
        sr = model.predict_step(tensor)
    sr_np = sr.squeeze(0).detach().cpu().numpy()

    output_profile.update(
        count=sr_np.shape[0],
        dtype='float32',
        height=sr_np.shape[1],
        width=sr_np.shape[2],
        transform=source_transform * source_transform.scale(
            source_width / sr_np.shape[2], source_height / sr_np.shape[1]
        ),
    )
    output_path = Path(output_tif)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(output_path, 'w', **output_profile) as dst:
        dst.write(np.clip(sr_np, 0, 1).astype(np.float32))

    print(f'Wrote super-resolved GeoTIFF to {output_path}')
    print(f'Output shape: {sr_np.shape}')
    return output_path

output_path = run_inference(INPUT_TIF, OUTPUT_TIF, model_input, model, device)

## 6. Verify the output

In [ ]:
with rasterio.open(output_path) as src:
    output_data = src.read()
    print(f'File   : {output_path}')
    print(f'Size   : {src.width} x {src.height}')
    print(f'Bands  : {src.count}')
    print(f'Dtype  : {src.dtypes[0]}')
    print(f'CRS    : {src.crs}')
    print(f'Transform: {src.transform}')
    print(f'Range : {output_data.min():.6f} to {output_data.max():.6f}')

assert src.count == 4, f'Expected 4 output bands, found {src.count}'
assert src.width >= 4 * (model_input.shape[2] - 1), 'Output width is not approximately 4x the input width'
assert src.height >= 4 * (model_input.shape[1] - 1), 'Output height is not approximately 4x the input height'
assert src.crs is not None, 'Output CRS is missing'
assert np.isfinite(output_data).all(), 'Output contains non-finite values'

The output remains in Google Drive at `OUTPUT_TIF`. Download it from the Files panel or use the optional cell below.

In [ ]:
# Optional: download the result directly from Colab.
# from google.colab import files
# files.download(str(output_path))